# Week 3 – Feature Engineering

In this notebook, time-based and lag features are created
to improve model learning for energy forecasting.


In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("smart_home_energy_consumption_large.csv")

# Combine Date and Time
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df = df.set_index('Datetime')

# Drop unused columns
df = df.drop(columns=['Date', 'Time'])

df.head()


,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size
Datetime,,,,,,
2023-12-02 21:12:00,94,Fridge,0.20,-1.0,Fall,2
2023-08-06 20:11:00,435,Oven,0.23,31.1,Summer,5
2023-11-21 06:39:00,466,Dishwasher,0.32,21.3,Fall,3
2023-01-21 21:56:00,496,Heater,3.92,-4.2,Winter,1
2023-08-26 04:31:00,137,Microwave,0.44,34.5,Summer,5


In [2]:
# Extract time-based features
df['hour'] = df.index.hour
df['day'] = df.index.day
df['weekday'] = df.index.weekday
df['month'] = df.index.month

df.head()


,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,hour,day,weekday,month
Datetime,,,,,,,,,,
2023-12-02 21:12:00,94,Fridge,0.20,-1.0,Fall,2,21,2,5,12
2023-08-06 20:11:00,435,Oven,0.23,31.1,Summer,5,20,6,6,8
2023-11-21 06:39:00,466,Dishwasher,0.32,21.3,Fall,3,6,21,1,11
2023-01-21 21:56:00,496,Heater,3.92,-4.2,Winter,1,21,21,5,1
2023-08-26 04:31:00,137,Microwave,0.44,34.5,Summer,5,4,26,5,8


In [3]:
energy_col = "Energy Consumption (kWh)"

# Sort by datetime to ensure proper lag
df = df.sort_index()

# Create lag features
df['lag_1'] = df[energy_col].shift(1)
df['lag_24'] = df[energy_col].shift(24)

df.head()


,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,hour,day,weekday,month,lag_1,lag_24
Datetime,,,,,,,,,,,,
2023-01-01 00:07:00,140,Lights,1.00,-5.9,Winter,1,0,1,6,1,NaN,NaN
2023-01-01 00:13:00,298,Lights,1.09,22.0,Winter,1,0,1,6,1,1.00,NaN
2023-01-01 00:24:00,469,Fridge,0.30,-1.2,Winter,3,0,1,6,1,1.09,NaN
2023-01-01 00:26:00,398,Fridge,0.50,35.8,Winter,2,0,1,6,1,0.30,NaN
2023-01-01 00:30:00,293,Washing Machine,1.12,2.7,Winter,3,0,1,6,1,0.50,NaN


In [4]:
# Create rolling mean feature
df['rolling_mean_24'] = df[energy_col].rolling(window=24).mean()

df.head()


,Home ID,Appliance Type,Energy Consumption (kWh),Outdoor Temperature (°C),Season,Household Size,hour,day,weekday,month,lag_1,lag_24,rolling_mean_24
Datetime,,,,,,,,,,,,,
2023-01-01 00:07:00,140,Lights,1.00,-5.9,Winter,1,0,1,6,1,NaN,NaN,NaN
2023-01-01 00:13:00,298,Lights,1.09,22.0,Winter,1,0,1,6,1,1.00,NaN,NaN
2023-01-01 00:24:00,469,Fridge,0.30,-1.2,Winter,3,0,1,6,1,1.09,NaN,NaN
2023-01-01 00:26:00,398,Fridge,0.50,35.8,Winter,2,0,1,6,1,0.30,NaN,NaN
2023-01-01 00:30:00,293,Washing Machine,1.12,2.7,Winter,3,0,1,6,1,0.50,NaN,NaN


In [5]:
df = df.dropna()

print("After Feature Engineering:")
print(df.head())


After Feature Engineering:
                     Home ID    Appliance Type  Energy Consumption (kWh)  \
Datetime                                                                   
2023-01-01 01:46:00       60            Fridge                      0.21   
2023-01-01 01:46:00      475  Air Conditioning                      3.24   
2023-01-01 02:02:00      237   Washing Machine                      1.67   
2023-01-01 02:05:00      405            Lights                      0.38   
2023-01-01 02:05:00      140            Lights                      0.36   

                     Outdoor Temperature (°C)  Season  Household Size  hour  \
Datetime                                                                      
2023-01-01 01:46:00                       0.1  Winter               5     1   
2023-01-01 01:46:00                      13.0  Winter               5     1   
2023-01-01 02:02:00                      -1.6  Winter               4     2   
2023-01-01 02:05:00                       3.9